In [1]:
import os
import pandas as pd

from rdkit import Chem
from wombat.krfp_models import krfp_models

In [2]:
base_dir_shap_100 = "../artifacts/explainability_method_tester/R2/"
base_dir_shap_500 = "../artifacts/explainability_method_tester/R2-SHAP500/"

In [3]:
aurocs_100 = []
aurocs_500 = []

for subdir in os.listdir(base_dir_shap_100):
    df_100 = pd.read_csv(f"{base_dir_shap_100}/{subdir}/positive_explainability_results.csv")
    df_500 = pd.read_csv(f"{base_dir_shap_500}/{subdir}/positive_explainability_results.csv")

    df_100 = df_100.dropna(subset=["SHAP Sampling_auroc"])  # Drop rows where SHAP Sampling_auroc is NaN
    df_500 = df_500.dropna(subset=["SHAP Sampling_auroc"])  # Drop rows where SHAP Sampling_auroc is NaN

    shap_100 = df_100["SHAP Sampling_auroc"] == 1.0
    shap_500 = df_500["SHAP Sampling_auroc"] == 1.0

    mean_100 = shap_100.mean()
    mean_500 = shap_500.mean()

    aurocs_100.append(mean_100)
    aurocs_500.append(mean_500)

latex_table_df = pd.DataFrame({
    "Whitebox": os.listdir(base_dir_shap_100),
    "SHAP 100": aurocs_100,
    "SHAP 500": aurocs_500
})

latex_table_df = latex_table_df.sort_values(by="Whitebox")

# Reset the index
latex_table_df.reset_index(drop=True, inplace=True)

# Index name
latex_table_df.index.name = "Index"


In [4]:
entries = []

for _, pre_pattern, _ in krfp_models:
    mol = Chem.MolFromSmarts(pre_pattern)
    smiles = Chem.MolToSmiles(mol)
    smiles = smiles.replace("*", "C")
    mol = Chem.MolFromSmiles(smiles)

    entries.append(mol.GetNumHeavyAtoms())

latex_table_df["Pattern Size"] = entries

latex_table_df.to_csv("notebook_artifacts/shap_improvement.csv", index=True)
